**Purpose:**

Validate and document the alignment of the O*NET task universe with the Eloundou et al. exposure labels, identify the subset of tasks usable for exposure-score construction, and preserve the aligned task-level audit files for reproducibility.

### Eloundou Task-Level Exposure Source

The task-level exposure labels used in this analysis were obtained from the  
[Eloundou et al. `full_labelset.tsv` source file](https://github.com/openai/GPTs-are-GPTs/blob/main/data/full_labelset.tsv).

For reproducibility, the source file was downloaded once and preserved locally in this repository as:

`data/input/full_labelset.tsv`

All subsequent processing reads from this preserved local copy rather than retrieving the file from the live GitHub repository.

In [ ]:
# ------------------------------------------------------------------
# Step 1 — Load Eloundou et al. task-level exposure data
# ------------------------------------------------------------------
import pandas as pd
from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/GenAI-task-affordance-labor-outcomes"
)

ELOUNDOU_URL = (
    BASE_DIR
    / "06_Eloundou_Task_Alignment"
    / "Input"
    / "full_labelset.tsv"
)
eloundou_raw = pd.read_csv(
    ELOUNDOU_URL,
    sep="\t",
    index_col=0
)

# Working copy of Eloundou source
# Keep eloundou_raw untouched
eloundou = eloundou_raw.copy()

# Eloundou matching key
eloundou["task_id_match"] = (
    pd.to_numeric(eloundou["Task ID"], errors="coerce")
    .astype("Int64")
)
print(eloundou_raw.columns)

In [ ]:
# ------------------------------------------------------------------
# Step 2 — Load our frozen O*NET 30.0 task universe
# Purpose: prepare matching keys for our tasks and Eloundou tasks
# ------------------------------------------------------------------
OUR_TASK_FILE = (
    BASE_DIR
    / "02_Task_Data_and_Classification"
    / "Output"
    / "Occupational_Task_Universe_ONET30_v2.csv"
)
our_tasks = pd.read_csv(OUR_TASK_FILE)

# Our O*NET matching key
our_tasks["task_id_match"] = (
    pd.to_numeric(our_tasks["task_id"], errors="coerce")
    .astype("Int64")
)

# Working copy of Eloundou source
# Keep eloundou_raw untouched
eloundou = eloundou_raw.copy()

# Eloundou matching key
eloundou["task_id_match"] = (
    pd.to_numeric(eloundou["Task ID"], errors="coerce")
    .astype("Int64")
)

print("OUR TASK UNIVERSE")
print("Rows:", len(our_tasks))
print("Unique Task IDs:", our_tasks["task_id_match"].nunique())

print()

print("ELOUNDOU")
print("Rows:", len(eloundou))
print("Unique Task IDs:", eloundou["task_id_match"].nunique())

In [ ]:
# ------------------------------------------------------------------
# Step 3 — Define the Eloundou alignment universe
# Purpose: retain the complete frozen O*NET 30.0 task universe
# ------------------------------------------------------------------

alignment_tasks = our_tasks.copy()

assert len(alignment_tasks) == 13_508
assert alignment_tasks["task_id_match"].nunique() == 13_508
assert alignment_tasks["soc6"].nunique() == 604

print(f"Alignment task records: {len(alignment_tasks):,}")
print(f"SOC6 occupations: {alignment_tasks['soc6'].nunique():,}")

In [ ]:
# ------------------------------------------------------------------
# Step 4 — Apply direct Eloundou Task-ID alignment
# ------------------------------------------------------------------

eloundou_lookup = (
    eloundou[
        [
            "task_id_match",
            "O*NET-SOC Code",
            "Task",
            "Task Type",
            "human_exposure_agg",
            "gpt4_exposure",
        ]
    ]
    .rename(columns={
        "task_id_match": "eloundou_source_task_id",
        "O*NET-SOC Code": "eloundou_onet_soc_code",
        "Task": "eloundou_task_statement",
        "Task Type": "eloundou_task_type",
    })
)

assert eloundou_lookup["eloundou_source_task_id"].is_unique

aligned = alignment_tasks.merge(
    eloundou_lookup,
    left_on="task_id_match",
    right_on="eloundou_source_task_id",
    how="left",
    validate="one_to_one",
)

aligned["eloundou_match_type"] = (
    aligned["eloundou_source_task_id"]
    .notna()
    .map({
        True: "direct_task_id",
        False: "unmatched",
    })
)

print(aligned["eloundou_match_type"].value_counts())

In [ ]:
# ------------------------------------------------------------------
# Step 5 — Validate direct Task-ID matches
# ------------------------------------------------------------------

direct_audit = aligned.loc[
    aligned["eloundou_match_type"] == "direct_task_id"
].copy()

direct_audit["our_statement_key"] = (
    direct_audit["task_statement"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

direct_audit["eloundou_statement_key"] = (
    direct_audit["eloundou_task_statement"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

direct_audit["eloundou_soc6"] = (
    direct_audit["eloundou_onet_soc_code"]
    .astype("string")
    .str.extract(r"^(\d{2}-\d{4})", expand=False)
)

direct_audit["same_statement"] = (
    direct_audit["our_statement_key"]
    == direct_audit["eloundou_statement_key"]
)

direct_audit["same_soc6"] = (
    direct_audit["soc6"].astype("string")
    == direct_audit["eloundou_soc6"]
)

print(
    "Direct IDs with different statements:",
    (~direct_audit["same_statement"]).sum(),
)

print(
    "Direct IDs with different SOC6 codes:",
    (~direct_audit["same_soc6"]).sum(),
)

In [ ]:
# Relaxed comparison: ignore punctuation and spacing
direct_audit["our_statement_relaxed"] = (
    direct_audit["task_statement"]
    .astype("string")
    .str.lower()
    .str.replace(r"[^\w\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

direct_audit["eloundou_statement_relaxed"] = (
    direct_audit["eloundou_task_statement"]
    .astype("string")
    .str.lower()
    .str.replace(r"[^\w\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

material_candidates = direct_audit.loc[
    direct_audit["our_statement_relaxed"]
    != direct_audit["eloundou_statement_relaxed"]
].copy()

print(
    "Differences remaining after punctuation normalization:",
    len(material_candidates),
)

In [ ]:
# ------------------------------------------------------------------
# Step 6 — Inspect changed statements sharing the same Task ID
# ------------------------------------------------------------------

from difflib import SequenceMatcher

material_candidates["text_similarity"] = material_candidates.apply(
    lambda row: SequenceMatcher(
        None,
        row["our_statement_relaxed"],
        row["eloundou_statement_relaxed"],
    ).ratio(),
    axis=1,
)

statement_change_review = (
    material_candidates[
        [
            "soc6",
            "task_id",
            "task_statement",
            "eloundou_task_statement",
            "human_exposure_agg",
            "gpt4_exposure",
            "text_similarity",
        ]
    ]
    .sort_values("text_similarity")
)

with pd.option_context(
    "display.max_rows", 60,
    "display.max_colwidth", 200,
):
    display(statement_change_review)

In [ ]:
# ------------------------------------------------------------------
# Step 7 — Finalize direct Task-ID match categories
# ------------------------------------------------------------------

aligned["our_statement_key"] = (
    aligned["task_statement"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

aligned["eloundou_statement_key"] = (
    aligned["eloundou_task_statement"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

direct_mask = aligned["eloundou_source_task_id"].notna()

revised_statement_mask = (
    direct_mask
    & (
        aligned["our_statement_key"]
        != aligned["eloundou_statement_key"]
    )
)

aligned.loc[
    direct_mask & ~revised_statement_mask,
    "eloundou_match_type",
] = "direct_task_id_exact_statement"

aligned.loc[
    revised_statement_mask,
    "eloundou_match_type",
] = "direct_task_id_revised_statement"

print(aligned["eloundou_match_type"].value_counts())

In [ ]:
# ------------------------------------------------------------------
# Step 8 — Find cross-version statement matches for unmatched tasks
# ------------------------------------------------------------------

eloundou_statement_lookup = eloundou.copy()

eloundou_statement_lookup["eloundou_soc6"] = (
    eloundou_statement_lookup["O*NET-SOC Code"]
    .astype("string")
    .str.extract(r"^(\d{2}-\d{4})", expand=False)
)

eloundou_statement_lookup["statement_key"] = (
    eloundou_statement_lookup["Task"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

unmatched_tasks = (
    aligned.loc[
        aligned["eloundou_match_type"] == "unmatched",
        ["task_id_match", "soc6", "task_statement"],
    ]
    .rename(columns={"task_id_match": "our_task_id"})
    .copy()
)

unmatched_tasks["statement_key"] = (
    unmatched_tasks["task_statement"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
)

exact_candidates = unmatched_tasks.merge(
    eloundou_statement_lookup[
        [
            "task_id_match",
            "eloundou_soc6",
            "statement_key",
            "Task",
            "human_exposure_agg",
            "gpt4_exposure",
        ]
    ].rename(columns={
        "task_id_match": "eloundou_source_task_id",
        "Task": "eloundou_task_statement",
    }),
    left_on=["soc6", "statement_key"],
    right_on=["eloundou_soc6", "statement_key"],
    how="inner",
)

candidate_counts = (
    exact_candidates.groupby("our_task_id")[
        "eloundou_source_task_id"
    ]
    .nunique()
)

print("Unmatched tasks examined:", len(unmatched_tasks))
print("Unique exact statement matches:", (candidate_counts == 1).sum())
print("Multiple exact candidates:", (candidate_counts > 1).sum())
print(
    "No exact statement match:",
    len(unmatched_tasks) - len(candidate_counts),
)

In [ ]:
# ------------------------------------------------------------------
# Step 9 — Generate same-SOC6 semantic candidates
# ------------------------------------------------------------------

from difflib import SequenceMatcher

def relaxed_text(series):
    return (
        series.astype("string")
        .str.lower()
        .str.replace(r"[^\w\s]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

unmatched_tasks["relaxed_statement"] = relaxed_text(
    unmatched_tasks["task_statement"]
)

eloundou_statement_lookup["relaxed_statement"] = relaxed_text(
    eloundou_statement_lookup["Task"]
)

candidate_rows = []

for _, task in unmatched_tasks.iterrows():
    candidate_pool = eloundou_statement_lookup.loc[
        eloundou_statement_lookup["eloundou_soc6"]
        == task["soc6"]
    ].copy()

    candidate_pool["text_similarity"] = (
        candidate_pool["relaxed_statement"]
        .map(
            lambda source_text: SequenceMatcher(
                None,
                task["relaxed_statement"],
                source_text,
            ).ratio()
        )
    )

    top_candidates = (
        candidate_pool
        .nlargest(3, "text_similarity")
        .assign(
            our_task_id=task["our_task_id"],
            our_task_statement=task["task_statement"],
        )
    )

    candidate_rows.append(top_candidates)

semantic_candidates = pd.concat(
    candidate_rows,
    ignore_index=True,
)

semantic_candidates["candidate_rank"] = (
    semantic_candidates.groupby("our_task_id")[
        "text_similarity"
    ]
    .rank(method="first", ascending=False)
    .astype(int)
)

candidate_review = (
    semantic_candidates[
        [
            "our_task_id",
            "our_task_statement",
            "candidate_rank",
            "task_id_match",
            "Task",
            "human_exposure_agg",
            "gpt4_exposure",
            "text_similarity",
        ]
    ]
    .rename(columns={
        "task_id_match": "eloundou_candidate_task_id",
        "Task": "eloundou_candidate_statement",
    })
    .sort_values(
        ["our_task_id", "candidate_rank"]
    )
)

with pd.option_context(
    "display.max_rows", 130,
    "display.max_colwidth", 160,
):
    display(candidate_review)

In [ ]:
# ------------------------------------------------------------------
# Step 8 — Audit Eloundou benchmark coverage by SOC6
# ------------------------------------------------------------------

aligned["eloundou_aligned"] = (
    aligned["eloundou_source_task_id"].notna()
)

aligned["benchmark_task_weight"] = (
    aligned["task_type"]
    .map({"Core": 2, "Supplemental": 1})
    .fillna(1)
    .astype(int)
)

aligned["matched_task_weight"] = (
    aligned["benchmark_task_weight"]
    .where(aligned["eloundou_aligned"], 0)
)

coverage_by_soc6 = (
    aligned.groupby("soc6")
    .agg(
        total_tasks=("task_id", "size"),
        matched_tasks=("eloundou_aligned", "sum"),
        total_weight=("benchmark_task_weight", "sum"),
        matched_weight=("matched_task_weight", "sum"),
    )
    .reset_index()
)

coverage_by_soc6["task_coverage"] = (
    coverage_by_soc6["matched_tasks"]
    / coverage_by_soc6["total_tasks"]
)

coverage_by_soc6["weighted_coverage"] = (
    coverage_by_soc6["matched_weight"]
    / coverage_by_soc6["total_weight"]
)

affected_occupations = coverage_by_soc6.loc[
    coverage_by_soc6["matched_tasks"]
    < coverage_by_soc6["total_tasks"]
].sort_values("weighted_coverage")

print(
    "SOC6 occupations containing unmatched tasks:",
    len(affected_occupations),
)

display(affected_occupations)

In [ ]:
# ------------------------------------------------------------------
# Step 9 — Verify Eloundou label availability
# ------------------------------------------------------------------

matched_alignment = aligned.loc[
    aligned["eloundou_aligned"]
].copy()

print(f"Matched task records: {len(matched_alignment):,}")

print(
    "Missing human exposure labels:",
    matched_alignment["human_exposure_agg"].isna().sum(),
)

print(
    "Missing GPT-4 exposure labels:",
    matched_alignment["gpt4_exposure"].isna().sum(),
)

print("\nHuman exposure labels:")
print(
    matched_alignment["human_exposure_agg"]
    .value_counts(dropna=False)
)

print("\nGPT-4 exposure labels:")
print(
    matched_alignment["gpt4_exposure"]
    .value_counts(dropna=False)
)

In [ ]:
# ------------------------------------------------------------------
# Step 10 — Compare current and Eloundou task types
# ------------------------------------------------------------------

current_type = (
    matched_alignment["task_type"]
    .fillna("Missing")
)

eloundou_type = (
    matched_alignment["eloundou_task_type"]
    .fillna("Missing")
)

task_type_changed = current_type != eloundou_type

print(
    "Task-type differences:",
    task_type_changed.sum(),
)

print("\nCurrent versus Eloundou task type:")
print(
    pd.crosstab(
        current_type,
        eloundou_type,
        rownames=["O*NET 30.0"],
        colnames=["Eloundou source"],
        dropna=False,
    )
)

In [ ]:
# ------------------------------------------------------------------
# Step 11 — Create strict Eloundou benchmark task set
# ------------------------------------------------------------------

eloundou_usable = matched_alignment.copy()

eloundou_usable["eloundou_task_weight"] = (
    eloundou_usable["eloundou_task_type"]
    .map({
        "Core": 2,
        "Supplemental": 1,
    })
    .fillna(1)
    .astype(int)
)

assert len(eloundou_usable) == 13_466
assert eloundou_usable["human_exposure_agg"].notna().all()
assert eloundou_usable["gpt4_exposure"].notna().all()

print(
    eloundou_usable["eloundou_task_weight"]
    .value_counts()
    .sort_index()
)

print(
    "Total Eloundou task weight:",
    eloundou_usable["eloundou_task_weight"].sum(),
)

### SAVE THE AUDIT

In [ ]:
# ------------------------------------------------------------------
# Step 12 — Validate and save Eloundou alignment files
# ------------------------------------------------------------------

assert len(aligned) == 13_508
assert aligned["task_id_match"].is_unique
assert aligned["eloundou_aligned"].sum() == 13_466
assert len(eloundou_usable) == 13_466

output_path_aligned = (
    BASE_DIR
    / "06_Eloundou_Task_Alignment"
    / "Output"
    / "Occupational_Task_Eloundou_Alignment_ONET30.csv"
)

output_path_usable = (
    BASE_DIR
    / "06_Eloundou_Task_Alignment"
    / "Output"
    / "Occupational_Task_Eloundou_Exposure_Usable_v1.csv"
)

output_path_aligned.parent.mkdir(
    parents=True,
    exist_ok=True,
)

aligned.to_csv(
    output_path_aligned,
    index=False,
)

eloundou_usable.to_csv(
    output_path_usable,
    index=False,
)

print(f"Alignment audit records: {len(aligned):,}")
print(f"Benchmark-usable records: {len(eloundou_usable):,}")
print(f"Saved: {output_path_aligned}")
print(f"Saved: {output_path_usable}")